# Introducción a Regresión Logística con Scikit-Learn

**Elaborado por:** David Palacio J.  
**Correo:** davidpalacioj@gmail.com

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpalacioj/DataAI-Fundamentos-Aplicaciones/blob/main/InteligenciaArtificial/notebooks_teoria/09_1_intro_regresion_logistica.ipynb)

---

## 🎯 ¿Qué es la Regresión Logística?

La **regresión logística** es uno de los algoritmos más fundamentales para **clasificación** en Machine Learning. A pesar de su nombre, NO es para regresión sino para **predecir categorías o clases**.

### 📈 **Conceptos Clave:**

**🔢 Variable Dependiente (Y)**: Categoría que queremos predecir
- Clasificación binaria: 2 clases (Sí/No, Aprobado/Reprobado, Spam/No Spam)
- Clasificación multiclase: >2 clases (Alto/Medio/Bajo, Tipos de flores)

**📊 Variables Independientes (X)**: Las características que usamos para predecir
- Ejemplos: edad, ingresos, puntuación crediticia, características del email

**📐 La Función Sigmoide**:

La regresión logística transforma valores continuos en probabilidades entre 0 y 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Donde:
- $z = \beta_0 + \beta_1X_1 + \beta_2X_2 + ... + \beta_nX_n$
- **Salida**: Probabilidad entre 0 y 1
- **Decisión**: Si P(Y=1) > 0.5 → Clase 1, sino → Clase 0

## 🛠️ Instalación y Librerías

Vamos a usar las librerías más importantes para Machine Learning y visualización:

In [1]:
# Instalar librerías necesarias (solo en Colab)
# !pip install scikit-learn pandas numpy matplotlib seaborn plotly --quiet

# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-Learn para Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

# Configuración
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
np.random.seed(42)

print("✅ Librerías importadas exitosamente")

✅ Librerías importadas exitosamente


## 💳 Dataset: Aprobación de Tarjetas de Crédito

Vamos a crear un dataset sintético que simula la aprobación de tarjetas de crédito basándose en características del cliente.

### 📊 **Características del Dataset:**
- **1000 observaciones** de solicitudes de tarjeta
- **6 características** del cliente
- **Variable objetivo**: Aprobado (1) o Rechazado (0)

### 📝 **Nota:**
Usamos datos sintéticos para tener control total sobre las relaciones y patrones en los datos.

In [2]:
# Crear dataset sintético para clasificación binaria
np.random.seed(42)
n_samples = 1000

# Generar características realistas
edad = np.random.normal(40, 12, n_samples).clip(18, 70)
ingresos = np.random.exponential(50000, n_samples).clip(15000, 200000)
score_crediticio = np.random.normal(650, 100, n_samples).clip(300, 850)
años_empleo = np.random.exponential(5, n_samples).clip(0, 30)
num_tarjetas = np.random.poisson(2, n_samples).clip(0, 10)
historial_mora = np.random.binomial(1, 0.3, n_samples)

# Crear regla de aprobación (con algo de ruido)
probabilidad_aprobacion = (
    0.3 * (score_crediticio - 300) / 550 +
    0.3 * (ingresos - 15000) / 185000 +
    0.1 * (años_empleo / 30) +
    0.1 * (edad - 18) / 52 +
    0.1 * (1 - historial_mora) +
    0.1 * (1 - num_tarjetas / 10)
)

# Agregar ruido y convertir a binario
probabilidad_aprobacion += np.random.normal(0, 0.1, n_samples)
aprobado = (probabilidad_aprobacion > 0.5).astype(int)

# Crear DataFrame
df = pd.DataFrame({
    'edad': edad.round(0),
    'ingresos_anuales': ingresos.round(-2),
    'score_crediticio': score_crediticio.round(0),
    'años_empleo': años_empleo.round(1),
    'num_tarjetas_actuales': num_tarjetas,
    'historial_mora': historial_mora,
    'aprobado': aprobado
})

print(f"📊 Forma del dataset: {df.shape}")
print(f"📋 Columnas: {df.shape[1]}")
print(f"👥 Solicitudes: {df.shape[0]}")
print(f"\n✅ Tasa de aprobación: {df['aprobado'].mean():.1%}")
print(f"❌ Tasa de rechazo: {(1-df['aprobado'].mean()):.1%}")

print("\n📝 Primeras 5 solicitudes:")
df.head()

📊 Forma del dataset: (1000, 7)
📋 Columnas: 7
👥 Solicitudes: 1000

✅ Tasa de aprobación: 38.1%
❌ Tasa de rechazo: 61.9%

📝 Primeras 5 solicitudes:


,edad,ingresos_anuales,score_crediticio,años_empleo,num_tarjetas_actuales,historial_mora,aprobado
0,46.0,15000.0,619.0,1.5,2,0,0
1,38.0,15000.0,575.0,2.2,0,1,0
2,48.0,50600.0,682.0,5.6,1,0,0
3,58.0,61300.0,784.0,0.3,1,0,0
4,37.0,15000.0,462.0,1.2,2,1,0


### 📖 **Descripción de Variables**

| Variable | Descripción | Rango |
|----------|-------------|-------|
| **edad** | Edad del solicitante | 18-70 años |
| **ingresos_anuales** | Ingresos anuales en USD | $15k-$200k |
| **score_crediticio** | Puntuación crediticia FICO | 300-850 |
| **años_empleo** | Años en el empleo actual | 0-30 años |
| **num_tarjetas_actuales** | Número de tarjetas que ya tiene | 0-10 |
| **historial_mora** | Si ha tenido moras (1=Sí, 0=No) | 0 o 1 |
| **aprobado** | 🎯 **OBJETIVO**: Tarjeta aprobada | 1=Sí, 0=No |

In [3]:
# Estadísticas descriptivas por clase
print("📊 Estadísticas por grupo:")
print("\n✅ Clientes APROBADOS:")
print(df[df['aprobado']==1][['edad', 'ingresos_anuales', 'score_crediticio']].describe().round(0))

print("\n❌ Clientes RECHAZADOS:")
print(df[df['aprobado']==0][['edad', 'ingresos_anuales', 'score_crediticio']].describe().round(0))

📊 Estadísticas por grupo:

✅ Clientes APROBADOS:
        edad  ingresos_anuales  score_crediticio
count  381.0             381.0             381.0
mean    42.0           75540.0             682.0
std     11.0           53870.0              90.0
min     18.0           15000.0             397.0
25%     34.0           30700.0             617.0
50%     42.0           65100.0             689.0
75%     49.0          103300.0             740.0
max     70.0          200000.0             850.0

❌ Clientes RECHAZADOS:
        edad  ingresos_anuales  score_crediticio
count  619.0             619.0             619.0
mean    39.0           36676.0             631.0
std     12.0           28588.0              93.0
min     18.0           15000.0             348.0
25%     30.0           15000.0             566.0
50%     39.0           27000.0             632.0
75%     47.0           48550.0             698.0
max     70.0          200000.0             850.0


## 🔍 Análisis Exploratorio de Datos (EDA)

Antes de construir nuestro modelo, necesitamos entender las diferencias entre clientes aprobados y rechazados:

In [4]:
# Distribución de características por clase
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=['Score Crediticio', 'Ingresos Anuales', 'Edad',
                   'Años de Empleo', 'Número de Tarjetas', 'Historial de Mora']
)

columnas = ['score_crediticio', 'ingresos_anuales', 'edad', 
            'años_empleo', 'num_tarjetas_actuales', 'historial_mora']

for i, col in enumerate(columnas):
    row = i // 3 + 1
    col_idx = i % 3 + 1
    
    # Aprobados
    fig.add_trace(
        go.Box(y=df[df['aprobado']==1][col], name='Aprobado',
               marker_color='green', showlegend=(i==0)),
        row=row, col=col_idx
    )
    
    # Rechazados
    fig.add_trace(
        go.Box(y=df[df['aprobado']==0][col], name='Rechazado',
               marker_color='red', showlegend=(i==0)),
        row=row, col=col_idx
    )

fig.update_layout(
    title_text="Distribución de Características por Estado de Aprobación",
    height=600,
    showlegend=True
)
fig.show()

print("📊 Observaciones clave:")
print("  ✅ Los aprobados tienden a tener mayor score crediticio")
print("  ✅ Los aprobados generalmente tienen mayores ingresos")
print("  ❌ Los rechazados tienen más historial de mora")

📊 Observaciones clave:
  ✅ Los aprobados tienden a tener mayor score crediticio
  ✅ Los aprobados generalmente tienen mayores ingresos
  ❌ Los rechazados tienen más historial de mora


In [5]:
# Matriz de correlación
correlation_matrix = df.corr()

# Crear heatmap interactivo
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.round(2).values,
    texttemplate="%{text}",
    textfont={"size":10}
))

fig.update_layout(
    title="Matriz de Correlación - Aprobación de Tarjetas",
    height=500,
    width=700
)
fig.show()

# Variables más correlacionadas con la aprobación
approval_corr = correlation_matrix['aprobado'].abs().sort_values(ascending=False)[1:]
print("🎯 Variables más correlacionadas con la aprobación:")
for var, corr in approval_corr.items():
    signo = "➕" if correlation_matrix.loc[var, 'aprobado'] > 0 else "➖"
    print(f"  {signo} {var}: {corr:.3f}")

🎯 Variables más correlacionadas con la aprobación:
  ➕ ingresos_anuales: 0.426
  ➕ score_crediticio: 0.261
  ➖ historial_mora: 0.231
  ➕ edad: 0.136
  ➕ años_empleo: 0.111
  ➖ num_tarjetas_actuales: 0.029


## 🤖 Construyendo el Modelo de Regresión Logística

### 📊 **Paso 1: Preparar los Datos**

In [6]:
# Separar características (X) y variable objetivo (y)
X = df.drop('aprobado', axis=1)
y = df['aprobado']

print(f"📊 Forma de X (características): {X.shape}")
print(f"🎯 Forma de y (objetivo): {y.shape}")
print(f"\n📋 Características utilizadas: {list(X.columns)}")
print(f"\n🎯 Distribución de clases:")
print(f"  ✅ Aprobados: {y.sum()} ({y.mean():.1%})")
print(f"  ❌ Rechazados: {len(y) - y.sum()} ({1-y.mean():.1%})")

📊 Forma de X (características): (1000, 6)
🎯 Forma de y (objetivo): (1000,)

📋 Características utilizadas: ['edad', 'ingresos_anuales', 'score_crediticio', 'años_empleo', 'num_tarjetas_actuales', 'historial_mora']

🎯 Distribución de clases:
  ✅ Aprobados: 381 (38.1%)
  ❌ Rechazados: 619 (61.9%)


### 📊 **Paso 2: Dividir en Entrenamiento y Prueba**

In [7]:
# Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y  # Mantener proporción de clases
)

print("📊 División de datos completada:")
print(f"  🏋️ Entrenamiento: {X_train.shape[0]} solicitudes")
print(f"  🧪 Prueba: {X_test.shape[0]} solicitudes")
print(f"\n🎯 Proporción de aprobados:")
print(f"  🏋️ Entrenamiento: {y_train.mean():.1%}")
print(f"  🧪 Prueba: {y_test.mean():.1%}")

📊 División de datos completada:
  🏋️ Entrenamiento: 800 solicitudes
  🧪 Prueba: 200 solicitudes

🎯 Proporción de aprobados:
  🏋️ Entrenamiento: 38.1%
  🧪 Prueba: 38.0%


### 📊 **Paso 3: Escalar los Datos**

La regresión logística se beneficia de tener características en la misma escala:

In [8]:
# Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("📊 Escalado de características completado")
print("\n📈 Estadísticas después del escalado (entrenamiento):")
print(f"  Media: {X_train_scaled.mean():.3f}")
print(f"  Desv. Estándar: {X_train_scaled.std():.3f}")

📊 Escalado de características completado

📈 Estadísticas después del escalado (entrenamiento):
  Media: 0.000
  Desv. Estándar: 1.000


### 🤖 **Paso 4: Entrenar el Modelo**

In [9]:
# Crear y entrenar el modelo de regresión logística
modelo = LogisticRegression(random_state=42, max_iter=1000)

print("🚀 Entrenando modelo de regresión logística...")
modelo.fit(X_train_scaled, y_train)
print("✅ Modelo entrenado exitosamente!")

# Información sobre el modelo
print(f"\n📊 Intercepto (β₀): {modelo.intercept_[0]:.3f}")
print(f"📈 Número de coeficientes: {len(modelo.coef_[0])}")

🚀 Entrenando modelo de regresión logística...
✅ Modelo entrenado exitosamente!

📊 Intercepto (β₀): -0.657
📈 Número de coeficientes: 6


### 📊 **Interpretando los Coeficientes**

In [10]:
# Crear DataFrame con coeficientes
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': modelo.coef_[0],
    'Impacto_Abs': np.abs(modelo.coef_[0])
}).sort_values('Impacto_Abs', ascending=False)

print("📊 Coeficientes del modelo (ordenados por impacto):")
for _, row in coeficientes.iterrows():
    variable = row['Variable']
    coef = row['Coeficiente']
    signo = "📈" if coef > 0 else "📉"
    print(f"  {signo} {variable}: {coef:+.3f}")

# Visualizar coeficientes
fig = go.Figure()
colors = ['green' if coef > 0 else 'red' for coef in coeficientes['Coeficiente']]

fig.add_trace(go.Bar(
    x=coeficientes['Variable'],
    y=coeficientes['Coeficiente'],
    marker_color=colors,
    text=coeficientes['Coeficiente'].round(3),
    textposition='outside'
))

fig.update_layout(
    title="Coeficientes del Modelo de Regresión Logística",
    xaxis_title="Variables",
    yaxis_title="Coeficiente (impacto en log-odds)",
    height=400
)
fig.show()

print("\n💡 Interpretación:")
print("  📈 Coef. positivo → Aumenta probabilidad de aprobación")
print("  📉 Coef. negativo → Disminuye probabilidad de aprobación")

📊 Coeficientes del modelo (ordenados por impacto):
  📈 ingresos_anuales: +1.167
  📉 historial_mora: -0.761
  📈 score_crediticio: +0.748
  📈 edad: +0.364
  📈 años_empleo: +0.321
  📉 num_tarjetas_actuales: -0.113



💡 Interpretación:
  📈 Coef. positivo → Aumenta probabilidad de aprobación
  📉 Coef. negativo → Disminuye probabilidad de aprobación


## 🎯 Haciendo Predicciones y Evaluando el Modelo

### 🔮 **Paso 5: Hacer Predicciones**

In [11]:
# Hacer predicciones
y_train_pred = modelo.predict(X_train_scaled)
y_test_pred = modelo.predict(X_test_scaled)

# Probabilidades de predicción
y_train_proba = modelo.predict_proba(X_train_scaled)[:, 1]
y_test_proba = modelo.predict_proba(X_test_scaled)[:, 1]

print("🔮 Predicciones realizadas:")
print(f"  📊 Entrenamiento: {len(y_train_pred)} predicciones")
print(f"  🧪 Prueba: {len(y_test_pred)} predicciones")

# Mostrar algunas predicciones
comparacion = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicción': y_test_pred[:10],
    'Probabilidad': y_test_proba[:10],
    'Correcto': y_test.values[:10] == y_test_pred[:10]
})

print("\n📊 Primeras 10 predicciones:")
comparacion

🔮 Predicciones realizadas:
  📊 Entrenamiento: 800 predicciones
  🧪 Prueba: 200 predicciones

📊 Primeras 10 predicciones:


,Real,Predicción,Probabilidad,Correcto
0,0,0,0.111982,True
1,0,0,0.019381,True
2,0,0,0.195043,True
3,0,0,0.155689,True
4,1,1,0.896896,True
5,0,0,0.167442,True
6,0,0,0.042675,True
7,1,1,0.769966,True
8,0,0,0.108032,True
9,1,1,0.880843,True


### 📏 **Paso 6: Métricas de Evaluación**

In [12]:
# Calcular métricas
def calcular_metricas(y_real, y_pred, nombre_conjunto):
    accuracy = accuracy_score(y_real, y_pred)
    precision = precision_score(y_real, y_pred)
    recall = recall_score(y_real, y_pred)
    f1 = f1_score(y_real, y_pred)
    
    print(f"\n📊 Métricas para conjunto de {nombre_conjunto}:")
    print(f"  🎯 Accuracy (Exactitud): {accuracy:.3f}")
    print(f"  📏 Precision: {precision:.3f}")
    print(f"  📐 Recall (Sensibilidad): {recall:.3f}")
    print(f"  🔧 F1-Score: {f1:.3f}")
    
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1}

# Evaluar en ambos conjuntos
metricas_train = calcular_metricas(y_train, y_train_pred, "entrenamiento")
metricas_test = calcular_metricas(y_test, y_test_pred, "prueba")

# Comparación visual
metricas_df = pd.DataFrame({
    'Entrenamiento': list(metricas_train.values()),
    'Prueba': list(metricas_test.values())
}, index=['Accuracy', 'Precision', 'Recall', 'F1-Score'])

print("\n📋 Comparación de métricas:")
print(metricas_df.round(3))


📊 Métricas para conjunto de entrenamiento:
  🎯 Accuracy (Exactitud): 0.756
  📏 Precision: 0.724
  📐 Recall (Sensibilidad): 0.584
  🔧 F1-Score: 0.646

📊 Métricas para conjunto de prueba:
  🎯 Accuracy (Exactitud): 0.775
  📏 Precision: 0.731
  📐 Recall (Sensibilidad): 0.645
  🔧 F1-Score: 0.685

📋 Comparación de métricas:
           Entrenamiento  Prueba
Accuracy           0.756   0.775
Precision          0.724   0.731
Recall             0.584   0.645
F1-Score           0.646   0.685


### 📊 **Interpretación de Métricas de Clasificación:**

**🎯 Accuracy**: % de predicciones correctas totales
- Buena métrica general, pero puede engañar con clases desbalanceadas

**📏 Precision**: De los que predijimos como aprobados, ¿cuántos realmente lo fueron?
- Importante cuando el costo de falsos positivos es alto

**📐 Recall**: De todos los aprobados reales, ¿cuántos identificamos?
- Importante cuando el costo de falsos negativos es alto

**🔧 F1-Score**: Media armónica de Precision y Recall
- Balance entre ambas métricas

## 🎭 Matriz de Confusión

La matriz de confusión nos muestra exactamente dónde acierta y falla nuestro modelo:

In [13]:
# Calcular matrices de confusión
cm_train = confusion_matrix(y_train, y_train_pred)
cm_test = confusion_matrix(y_test, y_test_pred)

# Crear visualización
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Entrenamiento', 'Prueba']
)

# Matriz de entrenamiento
fig.add_trace(
    go.Heatmap(
        z=cm_train,
        x=['Pred: Rechazado', 'Pred: Aprobado'],
        y=['Real: Rechazado', 'Real: Aprobado'],
        colorscale='Blues',
        text=cm_train,
        texttemplate="%{text}",
        textfont={"size":16},
        showscale=False
    ),
    row=1, col=1
)

# Matriz de prueba
fig.add_trace(
    go.Heatmap(
        z=cm_test,
        x=['Pred: Rechazado', 'Pred: Aprobado'],
        y=['Real: Rechazado', 'Real: Aprobado'],
        colorscale='Blues',
        text=cm_test,
        texttemplate="%{text}",
        textfont={"size":16}
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Matrices de Confusión",
    height=400
)
fig.show()

# Análisis detallado para conjunto de prueba
tn, fp, fn, tp = cm_test.ravel()
print("\n📊 Análisis detallado (Conjunto de Prueba):")
print(f"  ✅ Verdaderos Positivos (TP): {tp} - Aprobados correctamente")
print(f"  ✅ Verdaderos Negativos (TN): {tn} - Rechazados correctamente")
print(f"  ❌ Falsos Positivos (FP): {fp} - Rechazados que deberían ser aprobados")
print(f"  ❌ Falsos Negativos (FN): {fn} - Aprobados que deberían ser rechazados")

print(f"\n💡 Interpretación para el negocio:")
print(f"  💳 Tarjetas aprobadas correctamente: {tp/(tp+fp):.1%}")
print(f"  🚫 Clientes buenos rechazados incorrectamente: {fp} ({fp/(tn+fp):.1%})")
print(f"  ⚠️ Clientes riesgosos aprobados incorrectamente: {fn} ({fn/(fn+tp):.1%})")


📊 Análisis detallado (Conjunto de Prueba):
  ✅ Verdaderos Positivos (TP): 49 - Aprobados correctamente
  ✅ Verdaderos Negativos (TN): 106 - Rechazados correctamente
  ❌ Falsos Positivos (FP): 18 - Rechazados que deberían ser aprobados
  ❌ Falsos Negativos (FN): 27 - Aprobados que deberían ser rechazados

💡 Interpretación para el negocio:
  💳 Tarjetas aprobadas correctamente: 73.1%
  🚫 Clientes buenos rechazados incorrectamente: 18 (14.5%)
  ⚠️ Clientes riesgosos aprobados incorrectamente: 27 (35.5%)


## 📈 Curva ROC y AUC

La curva ROC nos ayuda a visualizar el rendimiento del modelo en todos los umbrales posibles:

In [14]:
# Calcular curva ROC
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba)

auc_train = auc(fpr_train, tpr_train)
auc_test = auc(fpr_test, tpr_test)

# Crear gráfico
fig = go.Figure()

# Curva ROC entrenamiento
fig.add_trace(go.Scatter(
    x=fpr_train, y=tpr_train,
    mode='lines',
    name=f'Entrenamiento (AUC = {auc_train:.3f})',
    line=dict(color='blue')
))

# Curva ROC prueba
fig.add_trace(go.Scatter(
    x=fpr_test, y=tpr_test,
    mode='lines',
    name=f'Prueba (AUC = {auc_test:.3f})',
    line=dict(color='green')
))

# Línea diagonal (clasificador aleatorio)
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Clasificador Aleatorio',
    line=dict(color='red', dash='dash')
))

fig.update_layout(
    title='Curva ROC - Receiver Operating Characteristic',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR)',
    height=500,
    width=600
)
fig.show()

print("📊 Interpretación del AUC (Area Under Curve):")
print(f"  🎯 AUC Entrenamiento: {auc_train:.3f}")
print(f"  🎯 AUC Prueba: {auc_test:.3f}")
print("\n💡 Guía de interpretación:")
print("  0.9-1.0 = Excelente")
print("  0.8-0.9 = Muy bueno")
print("  0.7-0.8 = Bueno")
print("  0.6-0.7 = Regular")
print("  0.5-0.6 = Pobre")

📊 Interpretación del AUC (Area Under Curve):
  🎯 AUC Entrenamiento: 0.832
  🎯 AUC Prueba: 0.846

💡 Guía de interpretación:
  0.9-1.0 = Excelente
  0.8-0.9 = Muy bueno
  0.7-0.8 = Bueno
  0.6-0.7 = Regular
  0.5-0.6 = Pobre


## 🎨 Visualización de la Frontera de Decisión

Vamos a visualizar cómo el modelo separa las clases usando las dos características más importantes:

In [15]:
# Seleccionar las dos características más importantes
features_importantes = coeficientes.head(2)['Variable'].values
idx_features = [list(X.columns).index(f) for f in features_importantes]

# Entrenar modelo simplificado con solo 2 características
X_train_2d = X_train_scaled[:, idx_features]
X_test_2d = X_test_scaled[:, idx_features]

modelo_2d = LogisticRegression(random_state=42)
modelo_2d.fit(X_train_2d, y_train)

# Crear mesh para visualización
h = 0.02
x_min, x_max = X_test_2d[:, 0].min() - 1, X_test_2d[:, 0].max() + 1
y_min, y_max = X_test_2d[:, 1].min() - 1, X_test_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predecir probabilidades para cada punto del mesh
Z = modelo_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z = Z.reshape(xx.shape)

# Crear visualización
fig = go.Figure()

# Contorno de probabilidades
fig.add_trace(go.Contour(
    x=xx[0],
    y=yy[:, 0],
    z=Z,
    colorscale='RdBu',
    opacity=0.4,
    showscale=True,
    colorbar=dict(title="Probabilidad")
))

# Puntos de prueba
colors_test = ['red' if y == 0 else 'green' for y in y_test]
symbols_test = ['circle' if y == 0 else 'diamond' for y in y_test]

for clase, color, symbol, nombre in [(0, 'red', 'circle', 'Rechazado'), 
                                      (1, 'green', 'diamond', 'Aprobado')]:
    mask = y_test == clase
    fig.add_trace(go.Scatter(
        x=X_test_2d[mask, 0],
        y=X_test_2d[mask, 1],
        mode='markers',
        name=nombre,
        marker=dict(color=color, size=8, symbol=symbol,
                   line=dict(color='black', width=1))
    ))

# Línea de decisión (probabilidad = 0.5)
fig.add_trace(go.Contour(
    x=xx[0],
    y=yy[:, 0],
    z=Z,
    contours=dict(
        start=0.5,
        end=0.5,
        size=0.5
    ),
    line=dict(color='black', width=2),
    showscale=False,
    name='Frontera de Decisión'
))

fig.update_layout(
    title=f'Frontera de Decisión - {features_importantes[0]} vs {features_importantes[1]}',
    xaxis_title=f'{features_importantes[0]} (escalado)',
    yaxis_title=f'{features_importantes[1]} (escalado)',
    height=500
)
fig.show()

print("📊 Interpretación de la visualización:")
print("  🔵 Zonas azules: Alta probabilidad de rechazo")
print("  🔴 Zonas rojas: Alta probabilidad de aprobación")
print("  ⚫ Línea negra: Frontera de decisión (P=0.5)")
print(f"\n  Accuracy del modelo 2D: {modelo_2d.score(X_test_2d, y_test):.3f}")

📊 Interpretación de la visualización:
  🔵 Zonas azules: Alta probabilidad de rechazo
  🔴 Zonas rojas: Alta probabilidad de aprobación
  ⚫ Línea negra: Frontera de decisión (P=0.5)

  Accuracy del modelo 2D: 0.780


## 💳 Ejemplo Práctico: Evaluar una Nueva Solicitud

Vamos a usar nuestro modelo para evaluar una nueva solicitud de tarjeta:

In [16]:
# Definir características de un nuevo solicitante
nuevo_cliente = {
    'edad': 35,
    'ingresos_anuales': 75000,
    'score_crediticio': 720,
    'años_empleo': 8,
    'num_tarjetas_actuales': 2,
    'historial_mora': 0
}

# Convertir a DataFrame y escalar
cliente_df = pd.DataFrame([nuevo_cliente])
cliente_scaled = scaler.transform(cliente_df)

# Hacer predicción
prediccion = modelo.predict(cliente_scaled)[0]
probabilidad = modelo.predict_proba(cliente_scaled)[0]

print("👤 Características del nuevo solicitante:")
for caracteristica, valor in nuevo_cliente.items():
    print(f"  📊 {caracteristica}: {valor}")

print(f"\n🔮 Resultado de la evaluación:")
print(f"  📋 Decisión: {'✅ APROBADO' if prediccion == 1 else '❌ RECHAZADO'}")
print(f"  📊 Probabilidad de aprobación: {probabilidad[1]:.1%}")
print(f"  📊 Probabilidad de rechazo: {probabilidad[0]:.1%}")

# Análisis de factores
print(f"\n📈 Análisis de factores:")
contribuciones = modelo.coef_[0] * cliente_scaled[0]
factores_df = pd.DataFrame({
    'Factor': X.columns,
    'Valor_Cliente': cliente_df.values[0],
    'Contribución': contribuciones
}).sort_values('Contribución', ascending=False)

print("\n  Factores positivos (aumentan aprobación):")
for _, row in factores_df[factores_df['Contribución'] > 0].head(3).iterrows():
    print(f"    ✅ {row['Factor']}: +{row['Contribución']:.3f}")

print("\n  Factores negativos (disminuyen aprobación):")
for _, row in factores_df[factores_df['Contribución'] < 0].head(3).iterrows():
    print(f"    ⚠️ {row['Factor']}: {row['Contribución']:.3f}")

👤 Características del nuevo solicitante:
  📊 edad: 35
  📊 ingresos_anuales: 75000
  📊 score_crediticio: 720
  📊 años_empleo: 8
  📊 num_tarjetas_actuales: 2
  📊 historial_mora: 0

🔮 Resultado de la evaluación:
  📋 Decisión: ✅ APROBADO
  📊 Probabilidad de aprobación: 74.5%
  📊 Probabilidad de rechazo: 25.5%

📈 Análisis de factores:

  Factores positivos (aumentan aprobación):
    ✅ ingresos_anuales: +0.638
    ✅ score_crediticio: +0.554
    ✅ historial_mora: +0.470

  Factores negativos (disminuyen aprobación):
    ⚠️ num_tarjetas_actuales: -0.004
    ⚠️ edad: -0.171


## 🔄 Ajustando el Umbral de Decisión

Por defecto, el umbral es 0.5, pero podemos ajustarlo según las necesidades del negocio:

In [17]:
# Evaluar diferentes umbrales
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]
resultados_umbrales = []

for umbral in umbrales:
    y_pred_umbral = (y_test_proba >= umbral).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred_umbral)
    precision = precision_score(y_test, y_pred_umbral)
    recall = recall_score(y_test, y_pred_umbral)
    
    # Calcular tasas de aprobación
    tasa_aprobacion = y_pred_umbral.mean()
    
    resultados_umbrales.append({
        'Umbral': umbral,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'Tasa_Aprobación': tasa_aprobacion
    })

umbrales_df = pd.DataFrame(resultados_umbrales)

# Visualizar impacto del umbral
fig = go.Figure()

metricas = ['Accuracy', 'Precision', 'Recall', 'Tasa_Aprobación']
colores = ['blue', 'green', 'red', 'purple']

for metrica, color in zip(metricas, colores):
    fig.add_trace(go.Scatter(
        x=umbrales_df['Umbral'],
        y=umbrales_df[metrica],
        mode='lines+markers',
        name=metrica,
        line=dict(color=color, width=2),
        marker=dict(size=8)
    ))

fig.update_layout(
    title='Impacto del Umbral de Decisión en las Métricas',
    xaxis_title='Umbral de Decisión',
    yaxis_title='Valor de la Métrica',
    height=400,
    hovermode='x unified'
)
fig.show()

print("📊 Análisis de umbrales:")
print(umbrales_df.round(3))

print("\n💡 Recomendaciones según objetivo del negocio:")
print("  🎯 Maximizar accuracy: Umbral = 0.5")
print("  💰 Ser conservador (menos riesgo): Umbral = 0.6-0.7")
print("  📈 Capturar más clientes: Umbral = 0.3-0.4")

📊 Análisis de umbrales:
   Umbral  Accuracy  Precision  Recall  Tasa_Aprobación
0     0.3     0.750      0.625   0.855            0.520
1     0.4     0.755      0.659   0.737            0.425
2     0.5     0.775      0.731   0.645            0.335
3     0.6     0.795      0.830   0.579            0.265
4     0.7     0.785      0.923   0.474            0.195

💡 Recomendaciones según objetivo del negocio:
  🎯 Maximizar accuracy: Umbral = 0.5
  💰 Ser conservador (menos riesgo): Umbral = 0.6-0.7
  📈 Capturar más clientes: Umbral = 0.3-0.4


## 🎓 Resumen y Conceptos Clave

### 📚 **Lo que Aprendimos:**

1. **🎯 Regresión Logística**: Algoritmo de clasificación que predice probabilidades
2. **📊 Función Sigmoide**: Transforma valores lineales en probabilidades (0-1)
3. **🔍 Métricas de Clasificación**: Accuracy, Precision, Recall, F1-Score
4. **🎭 Matriz de Confusión**: Visualiza aciertos y errores del modelo
5. **📈 Curva ROC y AUC**: Evalúa el rendimiento en todos los umbrales
6. **⚖️ Trade-offs**: Balance entre precision y recall según el negocio

### 💡 **Cuándo Usar Regresión Logística:**

**✅ Buena opción cuando:**
- Problema de clasificación binaria o multiclase
- Necesitas probabilidades, no solo clases
- Requieres interpretabilidad de resultados
- Relaciones aproximadamente lineales
- Dataset pequeño/mediano

**❌ Considera otras opciones cuando:**
- Relaciones muy no lineales
- Interacciones complejas entre variables
- Datasets muy grandes (deep learning)
- Necesitas máxima precisión (ensemble methods)

### 🚀 **Próximos Pasos Recomendados:**

1. **🔧 Regularización**: L1 (Lasso) y L2 (Ridge) para evitar overfitting
2. **📊 Ingeniería de características**: Crear variables más predictivas
3. **🤖 Algoritmos avanzados**: Random Forest, XGBoost, Neural Networks
4. **📈 Validación cruzada**: Evaluación más robusta del modelo
5. **⚖️ Manejo de desbalance**: SMOTE, class weights, resampling

### 📖 **Recursos Adicionales:**

- [Documentación Scikit-Learn - Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- [Kaggle Learn - Classification](https://www.kaggle.com/learn/intro-to-machine-learning)
- [Interpretación de Métricas de Clasificación](https://towardsdatascience.com/understanding-confusion-matrix-a9ad42dcfd62)

---

**¡Ahora tienes las bases sólidas para hacer clasificación con Machine Learning! 🎯**